In [ ]:
# ── 1. Imports ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, precision_score,
                             recall_score, f1_score)
import matplotlib.pyplot as plt

In [ ]:
# ── 2. Cargar dataset CSV ────────────────────────────────────
# Dataset: Pima Indians Diabetes
# 768 muestras | 8 features | Target: Outcome (0=No Diabético, 1=Diabético)
df = pd.read_csv('../dataset/diabetes.csv')
X = df.drop('Outcome', axis=1)
y = df['Outcome']
print('Shape:', df.shape)
print(df['Outcome'].value_counts())

In [ ]:
# ── 3. LabelEncoder (compatibilidad backend) ─────────────────
le = LabelEncoder()
le.fit([0, 1])

In [ ]:
# ── 4. Split ────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42)

In [ ]:
# ── 5. Escalar ──────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

In [ ]:
# ── 6. Modelo + búsqueda de hiperparámetros ─────────────────
mlp = MLPClassifier(max_iter=5000, random_state=42)

param_grid = {
    'hidden_layer_sizes': [(16,), (32,), (16, 8), (32, 16)],
    'activation'        : ['relu', 'tanh'],
    'alpha'             : [0.001, 0.01, 0.1]
}
# Capas acordes a los 8 features del dataset Pima

grid_search = RandomizedSearchCV(
    mlp, param_grid, cv=5, n_jobs=-1,
    scoring='accuracy', random_state=42, n_iter=10)
grid_search.fit(X_train_sc, y_train)

best_model = grid_search.best_estimator_
print('Mejores parámetros:', grid_search.best_params_)

In [ ]:
# ── 7. Métricas ─────────────────────────────────────────────
y_pred = best_model.predict(X_test_sc)

print('Accuracy :', accuracy_score(y_test, y_pred))
print('Precision:', precision_score(y_test, y_pred))
print('Recall   :', recall_score(y_test, y_pred))
print('F1-Score :', f1_score(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm,
                       display_labels=['No Diabético','Diabético']).plot()
plt.title('Red Neuronal Artificial — Diabetes')
plt.show()

In [ ]:
# ── 8. Guardar modelo + scaler ──────────────────────────────
with open('guardados/modelo_mlp.pkl',  'wb') as f: pickle.dump(best_model, f)
with open('guardados/scaler_mlp.pkl',  'wb') as f: pickle.dump(scaler, f)

print('✅ Modelo RNA guardado')